# Bias and Fairness

## Overview

Machine learning systems learn patterns from data, and when that data reflects historical or social inequities, the resulting models can reproduce or amplify them. **Fairness** in machine learning is the study of how to measure, understand, and reduce unwanted disparities in model behavior across groups of people. This notebook surveys the common sources of bias, the formal fairness metrics used to quantify disparity, mitigation strategies, and a key impossibility result that constrains what can be achieved.

---

## 1. Types of Bias

Bias can enter the pipeline at many stages. The following taxonomy is widely used:

- **Historical bias**: the world itself is biased, so even a perfect sample of reality encodes inequity (for example, historical hiring data reflecting past discrimination).
- **Representation bias**: the sampling process under-represents some groups, so the model has little signal about them.
- **Measurement bias**: the features or labels are noisy or systematically distorted proxies for the quantity of interest (for example, using arrests as a proxy for crime).
- **Aggregation bias**: a single model is fit across distinct subpopulations that actually need different models, so it fits none of them well.
- **Evaluation bias**: the benchmark or test set does not reflect the deployment population, hiding group-specific failures.
- **Deployment bias**: the system is used in a context different from the one it was designed and validated for.

---

## 2. Fairness Metrics

Let $A \in \{0, 1\}$ be a **protected attribute** (group membership), $Y \in \{0, 1\}$ the true label, $\hat{Y} \in \{0, 1\}$ the predicted label, and $S$ a continuous score.

**Demographic parity (statistical parity)** requires the positive prediction rate to be equal across groups:

$$P(\hat{Y} = 1 \mid A = a) = P(\hat{Y} = 1 \mid A = b) \quad \forall a, b$$

**Disparate impact ratio** is the ratio of positive rates between the disadvantaged and advantaged groups. The legal **80% rule** flags a problem when this ratio falls below 0.8:

$$DI = \frac{P(\hat{Y} = 1 \mid A = 0)}{P(\hat{Y} = 1 \mid A = 1)} \geq 0.8$$

**Equal opportunity** requires equal true positive rates (TPR) across groups:

$$P(\hat{Y} = 1 \mid A = a, Y = 1) = P(\hat{Y} = 1 \mid A = b, Y = 1)$$

**Equalized odds** strengthens this by requiring equal TPR and equal false positive rates (FPR):

$$P(\hat{Y} = 1 \mid A = a, Y = y) = P(\hat{Y} = 1 \mid A = b, Y = y), \quad y \in \{0, 1\}$$

**Predictive parity (calibration)** requires that, conditioned on a score, the probability of a positive outcome is the same across groups:

$$P(Y = 1 \mid S = s, A = a) = P(Y = 1 \mid S = s, A = b)$$

**Individual fairness** asks that similar individuals receive similar predictions: if $d(x_i, x_j)$ is small under a task-appropriate metric, then the predictions $f(x_i)$ and $f(x_j)$ should be close (Dwork et al., Fairness Through Awareness).

---

## 3. Bias Mitigation

Mitigation techniques are grouped by where they act in the pipeline:

- **Pre-processing**: transform the training data before fitting (reweighing samples, relabeling, learning fair representations) so that downstream models inherit less bias.
- **In-processing**: modify the learning algorithm itself, typically by adding fairness constraints or penalties to the objective (for example, reductions approaches and adversarial debiasing).
- **Post-processing**: adjust a trained model's outputs, for example by choosing group-specific decision thresholds to equalize a chosen metric.

---

## 4. The Impossibility Theorem

Chouldechova (2017) and Kleinberg et al. (2016) showed that several intuitive fairness criteria are mutually incompatible. Specifically, when base rates differ between groups ($P(Y = 1 \mid A = 0) \neq P(Y = 1 \mid A = 1)$), a classifier cannot simultaneously satisfy:

1. **Predictive parity** (equal positive predictive value across groups),
2. **Equal false positive rate**, and
3. **Equal false negative rate**.

Except in degenerate cases (perfect prediction or equal base rates), you must trade off among these properties. This means **fairness is not a single number**: choosing which criterion to satisfy is a value judgment about the application, not a purely technical decision.

---

## 5. Tooling

Open-source libraries package these metrics and mitigations: **Fairlearn**, **AIF360** (AI Fairness 360), and the **What-If Tool** for interactive model probing. Example calls appear in the commented optional block below.

---


In [3]:
# == Simulate a biased dataset (numpy only) =====================
import numpy as np

rng = np.random.default_rng(42)
n = 4000

# Protected attribute A in {0, 1}: group 1 is the advantaged group
A = rng.integers(0, 2, size=n)

# Base rates differ by group (this is what drives the impossibility result)
base_rate = np.where(A == 1, 0.55, 0.30)
Y = (rng.random(n) < base_rate).astype(int)

# A model score that is informative about Y but also leaks group membership,
# producing a biased scoring function.
signal = 1.4 * Y
group_shift = 0.6 * A          # advantaged group scored higher regardless of Y
noise = rng.normal(0, 1.0, size=n)
raw = signal + group_shift + noise
score = 1.0 / (1.0 + np.exp(-raw))   # squash to (0, 1)

print('n =', n)
print('overall positive label rate:', round(Y.mean(), 3))
for g in (0, 1):
    m = A == g
    print(f'group {g}: base rate = {Y[m].mean():.3f}, mean score = {score[m].mean():.3f}')


n = 4000
overall positive label rate: 0.441
group 0: base rate = 0.314, mean score = 0.581
group 1: base rate = 0.570, mean score = 0.747


In [4]:
# == Fairness metrics implemented from scratch ==================
def positive_rate(y_pred, mask):
    # P(Yhat = 1 | A = a)
    return y_pred[mask].mean()

def confusion_counts(y_true, y_pred, mask):
    yt, yp = y_true[mask], y_pred[mask]
    tp = int(np.sum((yp == 1) & (yt == 1)))
    fp = int(np.sum((yp == 1) & (yt == 0)))
    tn = int(np.sum((yp == 0) & (yt == 0)))
    fn = int(np.sum((yp == 0) & (yt == 1)))
    return tp, fp, tn, fn

def tpr_fpr(y_true, y_pred, mask):
    tp, fp, tn, fn = confusion_counts(y_true, y_pred, mask)
    tpr = tp / (tp + fn) if (tp + fn) else 0.0   # P(Yhat=1 | Y=1)
    fpr = fp / (fp + tn) if (fp + tn) else 0.0   # P(Yhat=1 | Y=0)
    return tpr, fpr

def disparate_impact(y_pred, A):
    # ratio of positive rates: disadvantaged (0) over advantaged (1)
    pr0 = positive_rate(y_pred, A == 0)
    pr1 = positive_rate(y_pred, A == 1)
    return pr0 / pr1 if pr1 else float('inf')

# Baseline decision: a single global threshold of 0.5
y_pred = (score >= 0.5).astype(int)

di = disparate_impact(y_pred, A)
print('Disparate impact ratio (0 / 1):', round(di, 3),
      '-> passes 80% rule' if di >= 0.8 else '-> FAILS 80% rule')
print()
for g in (0, 1):
    m = A == g
    tpr, fpr = tpr_fpr(Y, y_pred, m)
    print(f'group {g}: positive rate = {positive_rate(y_pred, m):.3f}, '
          f'TPR = {tpr:.3f}, FPR = {fpr:.3f}')


Disparate impact ratio (0 / 1): 0.716 -> FAILS 80% rule

group 0: positive rate = 0.624, TPR = 0.916, FPR = 0.490
group 1: positive rate = 0.871, TPR = 0.976, FPR = 0.732


In [5]:
# == Post-processing: per-group thresholds that equalize TPR ====
# Strategy: pick a target TPR, then for each group find the threshold whose
# achieved TPR is closest to that target. This is a simple post-processing
# mitigation that adjusts decision rules per group without retraining.
candidate_thresholds = np.linspace(0.05, 0.95, 181)
target_tpr = 0.70

def threshold_for_target_tpr(score, y_true, mask, thresholds, target):
    best_t, best_gap = 0.5, float('inf')
    for t in thresholds:
        yp = (score >= t).astype(int)
        tpr, _ = tpr_fpr(y_true, yp, mask)
        gap = abs(tpr - target)
        if gap < best_gap:
            best_gap, best_t = gap, t
    return best_t

thresholds_by_group = {}
for g in (0, 1):
    m = A == g
    thresholds_by_group[g] = threshold_for_target_tpr(
        score, Y, m, candidate_thresholds, target_tpr)

# Apply per-group thresholds
y_pred_fair = np.zeros_like(Y)
for g in (0, 1):
    m = A == g
    y_pred_fair[m] = (score[m] >= thresholds_by_group[g]).astype(int)

print('chosen thresholds:', {g: round(t, 3) for g, t in thresholds_by_group.items()})
print()
for g in (0, 1):
    m = A == g
    tpr, fpr = tpr_fpr(Y, y_pred_fair, m)
    print(f'group {g}: TPR = {tpr:.3f}, FPR = {fpr:.3f}')
print()
print('TPR gap before:',
      round(abs(tpr_fpr(Y, y_pred, A == 0)[0] - tpr_fpr(Y, y_pred, A == 1)[0]), 3))
print('TPR gap after: ',
      round(abs(tpr_fpr(Y, y_pred_fair, A == 0)[0] - tpr_fpr(Y, y_pred_fair, A == 1)[0]), 3))


chosen thresholds: {0: np.float64(0.695), 1: np.float64(0.81)}

group 0: TPR = 0.704, FPR = 0.211
group 1: TPR = 0.705, FPR = 0.207

TPR gap before: 0.06
TPR gap after:  0.001


In [6]:
# == Illustrating the impossibility theorem =====================
# Equalizing TPR (equal opportunity) does NOT generally equalize FPR or PPV
# when base rates differ. We print PPV (predictive parity) per group after
# the TPR-equalizing post-processing above.
def ppv(y_true, y_pred, mask):
    tp, fp, tn, fn = confusion_counts(y_true, y_pred, mask)
    return tp / (tp + fp) if (tp + fp) else 0.0

for g in (0, 1):
    m = A == g
    print(f'group {g}: PPV = {ppv(Y, y_pred_fair, m):.3f}, '
          f'FPR = {tpr_fpr(Y, y_pred_fair, m)[1]:.3f}')
print('\nBecause base rates differ (0.30 vs 0.55), PPV and FPR cannot both be')
print('matched while also matching TPR. This is the Chouldechova (2017) result.')


# == Optional: Fairlearn mitigation (requires extra installs) ===
# pip install fairlearn aif360
#
# from sklearn.linear_model import LogisticRegression
# from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
#
# X = score.reshape(-1, 1)
# base_estimator = LogisticRegression()
# mitigator = ExponentiatedGradient(
#     estimator=base_estimator,
#     constraints=EqualizedOdds(),
# )
# mitigator.fit(X, Y, sensitive_features=A)
# y_pred_mitigated = mitigator.predict(X)
#
# from fairlearn.metrics import MetricFrame, true_positive_rate, false_positive_rate
# mf = MetricFrame(
#     metrics={'tpr': true_positive_rate, 'fpr': false_positive_rate},
#     y_true=Y, y_pred=y_pred_mitigated, sensitive_features=A,
# )
# print(mf.by_group)


group 0: PPV = 0.604, FPR = 0.211
group 1: PPV = 0.818, FPR = 0.207

Because base rates differ (0.30 vs 0.55), PPV and FPR cannot both be
matched while also matching TPR. This is the Chouldechova (2017) result.


## Additional Learning Resources

### Papers

- Fairness Through Awareness (Dwork et al.) (https://arxiv.org/abs/1104.3913)
- Equality of Opportunity in Supervised Learning (Hardt et al.) (https://arxiv.org/abs/1610.02413)
- Inherent Trade-Offs in the Fair Determination of Risk Scores (Chouldechova) (https://arxiv.org/abs/1707.00046)

### Books & Courses

- Fairness and Machine Learning (Barocas, Hardt, Narayanan) (https://fairmlbook.org/)
- Google Responsible AI: Fairness & Bias (https://www.coursera.org/learn/responsible-ai-for-developers-fairness--bias)

### Tools & Libraries

- Fairlearn (https://github.com/fairlearn/fairlearn)
- AIF360 (https://github.com/Trusted-AI/AIF360)
- What-If Tool (https://github.com/pair-code/what-if-tool)
